In [ ]:
# use this notebook to download array data, subset it to kinship between 0.177–0.354 
# after this, run 2_methods_relatedness_R 

In [ ]:
import pandas as pd
import os
import numpy as np 
bucket=os.getenv('WORKSPACE_BUCKET')
import gcsfs

In [ ]:
# Set up GCSFileSystem with requester_pays=True
fs = gcsfs.GCSFileSystem(requester_pays=True)

gcs_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness.tsv"

with fs.open(gcs_path, 'r') as f:
    relatedness = pd.read_csv(f, sep = "\t")
    

In [ ]:
# subset relatedness to kinship between 0.177–0.354 
relatedness_par_sib = relatedness[(relatedness.kin <= 0.354) & (relatedness.kin >= 0.177)]

In [ ]:
#create a samples file with the two rows 
samples_list1 = relatedness_par_sib.loc[:,'i.s'].tolist()
samples_list2 = relatedness_par_sib.loc[:,'j.s'].tolist()
samples_list = samples_list1 + samples_list2
samples_list = list(set(samples_list))
samples_list = [str(x) for x in samples_list]
with open("sib_par.txt", "w") as file:
    for item in samples_list:
        file.write(f"0\t{item}\n")

In [ ]:
%%bash 
# copy array data
gsutil -m -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed $WORKSPACE_BUCKET/data/
gsutil -m -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim $WORKSPACE_BUCKET/data/
gsutil -m -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam $WORKSPACE_BUCKET/data/
gsutil -m -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed .
gsutil -m -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim .
gsutil -m -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam .

In [ ]:
%%bash 

# copy some relatedness files 
gsutil cp $WORKSPACE_BUCKET/relatedness/flagged_samples.tsv ./king_files/ 
gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness.tsv $WORKSPACE_BUCKET/king_files/relatedness.tsv
gsutil cp gs://fc-secure-f70bee07-e58f-4c5e-bc2a-47028421d402/bq_exports/linpoy@researchallofus.org/20250219/person_83609662/person_83609662_000000000000.csv $WORKSPACE_BUCKET/surveys/age_sex.txt 

In [ ]:
%%bash 

plink --bfile arrays \
  --keep sib_par.txt \
  --make-bed \
  --out sib_par

mkdir -p king_files 
mv sib_par* ./king_files

In [ ]:
%%bash 

# download and run king
wget https://www.kingrelatedness.com/executables/Linux-king216.tar.gz
tar -xzvf Linux-king216.tar.gz
./king -b ./king_files/sib_par.bed --related --prefix ./king_files/king_sib_par

In [ ]:
%%bash 

# split array data by chr

cd king_files 

for chr in {1..22}; do
  plink --bfile ./sib_par --chr $chr --make-bed --out sib_par_chr${chr}
done

In [ ]:
!mkdir -p relatedness 